In [ ]:
import torch

from main.model.downstream.fusion_probe.datamodule import FusionDataModule

In [ ]:
dm = FusionDataModule(seed=1, batch_size=8)
dm.add_dataset("/home/jacopo/dataset/EEGAVI/FUSION-DOWNSTREAM/DOWNSTREAM/interleaved-downstream", 1,
               valid_fraction=0.1)
dm.add_dataset("/home/jacopo/dataset/EEGAVI/FUSION-DOWNSTREAM/DOWNSTREAM/interleaved-downstream-dreamer", 1,
               valid_fraction=0.1)
dm.add_dataset("/home/jacopo/dataset/EEGAVI/FUSION-DOWNSTREAM/DOWNSTREAM/interleaved-downstream-deap", 1,
               test_fraction=1.0)

dm.setup("")

In [ ]:
len(dm.train_dataset) + len(dm.valid_dataset) + len(dm.test_dataset)

In [ ]:
len(dm.valid_dataset) + len(dm.test_dataset)

In [ ]:
len(dm.test_dataset)

In [ ]:
dl = dm.train_dataloader()
it = iter(dl)

scores_value = 0
score_count = 0
for i in it:
    try:
        scores = i["assessment", "scores"][:, 3]  # shape [B, T]
        valid = ~torch.isnan(scores)
        scores_value += scores[valid].sum().item()
        score_count += valid.sum().item()
    except:
        print(i["assessment", "scores"])

baseline = scores_value / score_count
print("baseline =", baseline)

In [ ]:
all_y = []

for batch in dm.val_dataloader():
    scores = batch["assessment", "scores"][:, 3]  # shape [B, T]
    valid = ~torch.isnan(scores)
    all_y.append(scores[valid])

y = torch.cat(all_y).float()

baseline_mse = ((y - baseline) ** 2).mean()
baseline_rmse = baseline_mse.sqrt()
baseline_mae = (y - baseline).abs().mean()

print("val_baseline_mse =", baseline_mse / 64)
print("val_baseline_rmse =", baseline_rmse / 8)
print("val_baseline_mae =", baseline_mae / 8)

In [ ]:
all_y = []

for batch in dm.test_dataloader():
    scores = batch["assessment", "scores"][:, 0]  # shape [B, T]
    valid = ~torch.isnan(scores)
    all_y.append(scores[valid])

y = torch.cat(all_y).float()

baseline_mse = ((y - baseline) ** 2).mean()
baseline_rmse = baseline_mse.sqrt()
baseline_mae = (y - baseline).abs().mean()

print("baseline_mse =", baseline_mse / 64)
print("baseline_rmse =", baseline_rmse / 8)
print("baseline_mae =", baseline_mae / 8)

In [ ]:
baseline_mse

Of data merged

In [9]:
import torch

from main.model.downstream.fusion_probe.datamodule import FusionDataModule

dm = FusionDataModule(seed=1, batch_size=8)
dm.add_dataset("/home/jacopo/dataset/EEGAVI/FUSION-DOWNSTREAM/DOWNSTREAM/interleaved-downstream", 1,
               valid_fraction=0.1, test_fraction=0.15)

print(len(dm.train_dataset[0]))
print(len(dm.valid_dataset[0]))
print(len(dm.test_dataset[0]))
# dm.add_dataset("/home/jacopo/dataset/EEGAVI/FUSION-DOWNSTREAM/DOWNSTREAM/interleaved-downstream-dreamer", 1,
#               valid_fraction=0.1, test_fraction=0.15)



dm.add_dataset("/home/jacopo/dataset/EEGAVI/FUSION-DOWNSTREAM/DOWNSTREAM/interleaved-downstream-deap", 1,
               valid_fraction=0.1, test_fraction=0.15)

print(len(dm.train_dataset[1]))
print(len(dm.valid_dataset[1]))
print(len(dm.test_dataset[1]))
dm.setup("")

519
76
112
306
36
72
306
36
72


In [7]:
519 +76 + 112

707

In [8]:
635 + 80 + 159

874

In [10]:
306 + 36 + 72

414

In [ ]:
dl = dm.train_dataloader()
it = iter(dl)
a = next(it)

In [ ]:
a["assessment", "scores"][:, 0, :]

In [2]:
sum_y = torch.zeros(3)
count_y = torch.zeros(3)

for batch in dm.train_dataloader():
    y = (batch["assessment", "scores"][:, 0, :].float() - 1) / 8
    valid = ~torch.isnan(y)

    y_zeroed = torch.where(valid, y, torch.zeros_like(y))
    sum_y += y_zeroed.sum(dim=0)
    count_y += valid.sum(dim=0)

baseline = sum_y / count_y
print("baseline =", baseline)  # shape [3]

baseline = tensor([0.5321, 0.5228, 0.5128])


In [3]:
se_sum = torch.zeros(3)
ae_sum = torch.zeros(3)
count_y = torch.zeros(3)

for batch in dm.val_dataloader():
    y = (batch["assessment", "scores"][:, 0, :].float() - 1) / 8
    valid = ~torch.isnan(y)

    err = y - baseline.unsqueeze(0)
    se_sum += torch.where(valid, err ** 2, torch.zeros_like(err)).sum(dim=0)
    ae_sum += torch.where(valid, err.abs(), torch.zeros_like(err)).sum(dim=0)
    count_y += valid.sum(dim=0)

mse = se_sum / count_y
rmse = mse.sqrt()
mae = ae_sum / count_y

print("val_baseline_mse =", mse)  # shape [3]
print("val_baseline_rmse =", rmse)  # shape [3]
print("val_baseline_mae =", mae)  # shape [3]

print("mean val_baseline_mse =", mse.mean())
print("mean val_baseline_rmse =", rmse.mean())
print("mean val_baseline_mae =", mae.mean())

val_baseline_mse = tensor([0.0620, 0.0736, 0.0809])
val_baseline_rmse = tensor([0.2489, 0.2714, 0.2845])
val_baseline_mae = tensor([0.2105, 0.2293, 0.2472])
mean val_baseline_mse = tensor(0.0722)
mean val_baseline_rmse = tensor(0.2683)
mean val_baseline_mae = tensor(0.2290)


In [4]:
se_sum = torch.zeros(3)
ae_sum = torch.zeros(3)
count_y = torch.zeros(3)

for batch in dm.test_dataloader():
    y = (batch["assessment", "scores"][:, 0, :].float() - 1) / 8
    valid = ~torch.isnan(y)

    err = y - baseline.unsqueeze(0)
    se_sum += torch.where(valid, err ** 2, torch.zeros_like(err)).sum(dim=0)
    ae_sum += torch.where(valid, err.abs(), torch.zeros_like(err)).sum(dim=0)
    count_y += valid.sum(dim=0)

mse = se_sum / count_y
rmse = mse.sqrt()
mae = ae_sum / count_y

print("val_baseline_mse =", mse)  # shape [3]
print("val_baseline_rmse =", rmse)  # shape [3]
print("val_baseline_mae =", mae)  # shape [3]

print("mean val_baseline_mse =", mse.mean())
print("mean val_baseline_rmse =", rmse.mean())
print("mean val_baseline_mae =", mae.mean())

val_baseline_mse = tensor([0.0418, 0.0641, 0.0416])
val_baseline_rmse = tensor([0.2044, 0.2532, 0.2040])
val_baseline_mae = tensor([0.1724, 0.2143, 0.1690])
mean val_baseline_mse = tensor(0.0492)
mean val_baseline_rmse = tensor(0.2205)
mean val_baseline_mae = tensor(0.1852)
